In [3]:
import json
import re
from pathlib import Path

# ============================================================
# V3A DATASET CONVERTER
# Old V3A -> Corrected V3A
# ============================================================

INPUT_FILE = "./dataset/V3A.jsonl"
OUTPUT_FILE = "./dataset/v3a_corrected.jsonl"

TASK_NAME = "v3a_user_profile_intelligence"

# Canonical V3A profile schema
PROFILE_FIELDS = [
    "profession",
    "career_goal",
    "education",
    "interests",
    "communication_preferences",
    "learning_style",
    "personality_traits",
    "lifestyle",
    "languages",
    "values",
    "social_preferences",
    "relationship_preferences",
    "work_preferences",
    "goals",
    "stable_habits",
    "preferences",
    "important_constraints",
    "confidence"
]


def empty_profile():
    """Create a clean V3A profile."""
    return {
        "profession": "",
        "career_goal": "",
        "education": "",
        "interests": [],
        "communication_preferences": [],
        "learning_style": "",
        "personality_traits": [],
        "lifestyle": [],
        "languages": [],
        "values": [],
        "social_preferences": [],
        "relationship_preferences": [],
        "work_preferences": [],
        "goals": [],
        "stable_habits": [],
        "preferences": [],
        "important_constraints": [],
        "confidence": 0.0
    }


def clean_string(value):
    if not isinstance(value, str):
        return ""

    value = value.strip()

    if value.lower() in [
        "unknown",
        "not specified",
        "not mentioned",
        "n/a",
        "none",
        "null"
    ]:
        return ""

    return value


def clean_list(value):
    if not isinstance(value, list):
        return []

    result = []

    for item in value:
        if isinstance(item, str):
            item = item.strip()

            if item and item.lower() not in [
                "unknown",
                "not specified",
                "not mentioned",
                "n/a",
                "none",
                "null"
            ]:
                result.append(item)

    return result


def find_user_statements(conversation):
    """
    Extract lines spoken by User.
    Used only for conservative evidence checking.
    """

    if not isinstance(conversation, str):
        return []

    statements = []

    for line in conversation.splitlines():

        line = line.strip()

        if line.lower().startswith("user:"):
            statements.append(
                line.split(":", 1)[1].strip()
            )

    return statements


def text_contains_any(text, terms):
    text = text.lower()

    return any(term.lower() in text for term in terms)


def remove_obvious_inference(profile, conversation, memories):
    """
    Conservative cleanup.

    This does NOT attempt to generate new facts.
    It mainly removes obvious unsupported/inferred values.
    """

    user_text = " ".join(find_user_statements(conversation))
    memory_text = " ".join(memories)

    evidence = f"{user_text} {memory_text}".lower()

    # --------------------------------------------------------
    # Profession
    # --------------------------------------------------------

    if profile["profession"]:

        profession = profile["profession"].lower()

        profession_terms = {
            "student": ["student", "studying", "degree", "university", "college"],
            "engineer": ["engineer", "engineering"],
            "designer": ["designer", "design"],
            "teacher": ["teacher", "teaching"],
            "researcher": ["researcher", "research"],
            "chef": ["chef"],
            "entrepreneur": ["entrepreneur", "business"],
            "translator": ["translator", "translation"],
            "psychologist": ["psychologist", "psychology"],
            "developer": ["developer", "programming", "software"],
            "artist": ["artist", "painting", "artist"],
        }

        supported = False

        for category, terms in profession_terms.items():

            if category in profession:

                if text_contains_any(evidence, terms):
                    supported = True

                break

        if not supported:
            profile["profession"] = ""

    # --------------------------------------------------------
    # Career goal
    # --------------------------------------------------------

    if profile["career_goal"]:

        goal = profile["career_goal"].lower()

        # Require meaningful evidence of future intention.
        future_words = [
            "goal",
            "hope",
            "want",
            "plan",
            "aim",
            "career",
            "future",
            "next year",
            "would like"
        ]

        if not any(word in evidence for word in future_words):
            profile["career_goal"] = ""

    # --------------------------------------------------------
    # Education
    # --------------------------------------------------------

    if profile["education"]:

        education_terms = [
            "student",
            "studying",
            "degree",
            "university",
            "college",
            "school",
            "master",
            "masters",
            "phd",
            "doctorate",
            "bachelor",
            "thesis",
            "graduat"
        ]

        if not any(term in evidence for term in education_terms):
            profile["education"] = ""

    # --------------------------------------------------------
    # Personality
    # Conservative: remove personality labels when there is
    # no repeated/clear behavioral evidence.
    # --------------------------------------------------------

    personality = profile["personality_traits"]

    if personality:

        if len(user_text) < 40 and len(memories) == 0:
            profile["personality_traits"] = []

    # --------------------------------------------------------
    # Health preferences
    #
    # Canonical V3A does NOT contain health_preferences.
    # They are intentionally discarded.
    # --------------------------------------------------------

    # Nothing to do because the field isn't copied.

    return profile


def convert_record(record):
    """
    Convert one old V3A example into canonical V3A format.
    """

    old_input = record.get("input", {})
    old_output = record.get("output", {})
    old_profile = old_output.get("profile", {})

    conversation = old_input.get("conversation", "")
    memories = old_input.get("user_memories", [])

    if not isinstance(memories, list):
        memories = []

    profile = empty_profile()

    # --------------------------------------------------------
    # Copy only canonical fields
    # --------------------------------------------------------

    field_aliases = {
        "communication_preferences": [
            "communication_preferences",
            "communication_preference"
        ]
    }

    for canonical_field in PROFILE_FIELDS:

        if canonical_field == "confidence":
            continue

        source_keys = field_aliases.get(
            canonical_field,
            [canonical_field]
        )

        value_found = None

        for key in source_keys:

            if key in old_profile:
                value_found = old_profile[key]
                break

        if value_found is None:
            continue

        if canonical_field in [
            "profession",
            "career_goal",
            "education",
            "learning_style"
        ]:
            profile[canonical_field] = clean_string(value_found)

        elif canonical_field in [
            "interests",
            "communication_preferences",
            "personality_traits",
            "lifestyle",
            "languages",
            "values",
            "social_preferences",
            "relationship_preferences",
            "work_preferences",
            "goals",
            "stable_habits",
            "preferences",
            "important_constraints"
        ]:
            profile[canonical_field] = clean_list(value_found)

    # --------------------------------------------------------
    # Conservative evidence cleanup
    # --------------------------------------------------------

    profile = remove_obvious_inference(
        profile,
        conversation,
        memories
    )

    # --------------------------------------------------------
    # Confidence
    # --------------------------------------------------------

    confidence = old_profile.get("confidence", 0.0)

    try:
        confidence = float(confidence)
    except (TypeError, ValueError):
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))

    profile["confidence"] = round(confidence, 2)

    # --------------------------------------------------------
    # Canonical V3A record
    # --------------------------------------------------------

    new_record = {
        "task": TASK_NAME,

        "instruction": (
            "Generate or update the user's long-term profile."
        ),

        "input": {
            "conversation": conversation,
            "conversation_summary": old_input.get(
                "conversation_summary",
                ""
            ),
            "conversation_understanding": old_input.get(
                "conversation_understanding",
                {}
            ),
            "user_memories": memories
        },

        "output": {
            "profile": profile
        }
    }

    return new_record


def convert_file():

    input_path = Path(INPUT_FILE)
    output_path = Path(OUTPUT_FILE)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file not found: {INPUT_FILE}"
        )

    total = 0
    converted = 0
    errors = 0

    with input_path.open(
        "r",
        encoding="utf-8"
    ) as infile, output_path.open(
        "w",
        encoding="utf-8"
    ) as outfile:

        for line_number, line in enumerate(infile, 1):

            line = line.strip()

            if not line:
                continue

            total += 1

            try:

                record = json.loads(line)

                new_record = convert_record(record)

                outfile.write(
                    json.dumps(
                        new_record,
                        ensure_ascii=False
                    ) + "\n"
                )

                converted += 1

            except Exception as e:

                errors += 1

                print(
                    f"[ERROR] Line {line_number}: {e}"
                )

    print("\n" + "=" * 60)
    print("V3A CONVERSION COMPLETE")
    print("=" * 60)

    print(f"Input records     : {total}")
    print(f"Converted records : {converted}")
    print(f"Errors            : {errors}")
    print(f"Output file       : {output_path}")
    print("=" * 60)


if __name__ == "__main__":
    convert_file()

[ERROR] Line 27: sequence item 0: expected str instance, dict found
[ERROR] Line 28: sequence item 0: expected str instance, dict found
[ERROR] Line 40: sequence item 0: expected str instance, dict found
[ERROR] Line 42: sequence item 0: expected str instance, dict found
[ERROR] Line 43: sequence item 0: expected str instance, dict found
[ERROR] Line 45: sequence item 0: expected str instance, dict found
[ERROR] Line 46: sequence item 0: expected str instance, dict found
[ERROR] Line 47: sequence item 0: expected str instance, dict found
[ERROR] Line 48: sequence item 0: expected str instance, dict found
[ERROR] Line 49: sequence item 0: expected str instance, dict found
[ERROR] Line 73: sequence item 0: expected str instance, dict found
[ERROR] Line 74: sequence item 0: expected str instance, dict found
[ERROR] Line 75: sequence item 0: expected str instance, dict found
[ERROR] Line 76: sequence item 0: expected str instance, dict found
[ERROR] Line 77: sequence item 0: expected str i